# GAN Evaluation

Compares generated vs real return distributions across three dimensions:
- **Left tail exceedance** — does the GAN capture extreme negative returns?
- **Center mass** — does the bulk of the distribution match?
- **Volatility** — are cross-asset volatilities and correlations preserved?

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

REPO_ROOT = Path("..")
sys.path.insert(0, str(REPO_ROOT))

from notebooks.TCN_marketgan_full_Lucas import (
    MarketGANConfig,
    prepare_marketgan_data,
    MarketGANSequenceDataset,
    MarketGenerator,
    MarketCritic,
    MarketGANTrainer,
)

CHECKPOINT_DIR = REPO_ROOT / "artifacts" / "marketgan_tcn" / "marketgan_proxy_factor_run" / "checkpoints"
N_SAMPLES = 50   # number of synthetic windows to generate
N_BOOTSTRAP = 500  # bootstrap iterations for confidence bands

print("Checkpoint dir:", CHECKPOINT_DIR)
print("Best checkpoint exists:", (CHECKPOINT_DIR / "best.pt").exists())

In [ ]:
# ── Load model and data ───────────────────────────────────────────────────────
config = MarketGANConfig(
    batch_size=16,
    target_horizon=252 * 4,
)

prepared_data = prepare_marketgan_data(config=config)
dataset = MarketGANSequenceDataset(prepared_data, sequence_length=config.total_sequence_length)

generator = MarketGenerator(prepared_data.dimensions, config).to(config.device)
critic    = MarketCritic(prepared_data.dimensions, config).to(config.device)
trainer   = MarketGANTrainer(generator=generator, critic=critic, config=config)

checkpoint = torch.load(CHECKPOINT_DIR / "best.pt", map_location=config.device)
generator.load_state_dict(checkpoint["generator_state_dict"])
generator.eval()

print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Assets : {prepared_data.dimensions.num_assets}")
print(f"Dataset windows: {len(dataset)}")

In [ ]:
# ── Generate N synthetic windows ──────────────────────────────────────────────
indices = np.linspace(0, len(dataset) - 1, N_SAMPLES, dtype=int)

generated_list = []
real_list      = []

with torch.no_grad():
    for idx in indices:
        sample = dataset[int(idx)]
        batch  = {k: v.unsqueeze(0).to(config.device) for k, v in sample.items()}

        out = generator(
            covariates=batch["covariates"],
            factor_returns=batch["factors"],
            alpha_hat=batch["alpha_hat"],
            beta_hat=batch["beta_hat"],
            sigma_hat=batch["sigma_hat"],
            trim_output=True,
        )
        # (target_horizon, n_assets)
        generated_list.append(out["generated_returns_trimmed"][0].cpu().numpy())
        real_list.append(batch["real_returns"][0, config.warmup_period:, :].cpu().numpy())

# Stack: (N_SAMPLES * target_horizon, n_assets)
gen_returns  = np.concatenate(generated_list, axis=0)
real_returns = np.concatenate(real_list,      axis=0)

asset_cols = prepared_data.asset_columns
gen_df  = pd.DataFrame(gen_returns,  columns=asset_cols)
real_df = pd.DataFrame(real_returns, columns=asset_cols)

print(f"Generated shape : {gen_df.shape}")
print(f"Real shape      : {real_df.shape}")

## 1. Center Mass — Distribution Comparison

In [ ]:
# Compare mean, std, skewness, kurtosis per asset
stats_real = pd.DataFrame({
    "mean":     real_df.mean(),
    "std":      real_df.std(),
    "skew":     real_df.skew(),
    "kurtosis": real_df.kurtosis(),
})

stats_gen = pd.DataFrame({
    "mean":     gen_df.mean(),
    "std":      gen_df.std(),
    "skew":     gen_df.skew(),
    "kurtosis": gen_df.kurtosis(),
})

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
metrics = ["mean", "std", "skew", "kurtosis"]
titles  = ["Mean return", "Volatility (std)", "Skewness", "Excess Kurtosis"]

for ax, metric, title in zip(axes.flat, metrics, titles):
    ax.scatter(stats_real[metric], stats_gen[metric], alpha=0.7, s=40)
    lims = [min(stats_real[metric].min(), stats_gen[metric].min()),
            max(stats_real[metric].max(), stats_gen[metric].max())]
    ax.plot(lims, lims, "r--", linewidth=1, label="perfect")
    ax.set_xlabel("Real")
    ax.set_ylabel("Generated")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Center Mass: Real vs Generated per Asset", fontsize=14)
plt.tight_layout()
plt.savefig(REPO_ROOT / "artifacts" / "eval_center_mass.png", dpi=150)
plt.show()

## 2. Left Tail Exceedance

In [ ]:
# For each threshold q, compute fraction of returns below q
# Bootstrap confidence bands from real data

thresholds = np.linspace(-0.10, -0.005, 60)  # -10% to -0.5% daily returns

def exceedance_curve(df, thresholds):
    """Fraction of all returns below each threshold (pooled across assets)"""
    flat = df.values.flatten()
    return np.array([(flat < t).mean() for t in thresholds])

# Bootstrap confidence bands on real data
flat_real = real_df.values.flatten()
boot_curves = []
for _ in range(N_BOOTSTRAP):
    sample = np.random.choice(flat_real, size=len(flat_real), replace=True)
    boot_curves.append(np.array([(sample < t).mean() for t in thresholds]))

boot_curves = np.array(boot_curves)
band_low  = np.percentile(boot_curves, 2.5,  axis=0)
band_high = np.percentile(boot_curves, 97.5, axis=0)

real_exc = exceedance_curve(real_df, thresholds)
gen_exc  = exceedance_curve(gen_df,  thresholds)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, real_exc, "b-",  linewidth=2, label="Real")
ax.plot(thresholds, gen_exc,  "r--", linewidth=2, label="Generated")
ax.fill_between(thresholds, band_low, band_high, alpha=0.2, color="blue", label="Real 95% CI (bootstrap)")
ax.set_xlabel("Return threshold")
ax.set_ylabel("Exceedance probability")
ax.set_title("Left Tail Exceedance: Real vs Generated")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(REPO_ROOT / "artifacts" / "eval_left_tail.png", dpi=150)
plt.show()

# KS test per asset
ks_results = []
for col in asset_cols:
    stat, pval = stats.ks_2samp(real_df[col].dropna(), gen_df[col].dropna())
    ks_results.append({"asset": col, "ks_stat": round(stat, 4), "p_value": round(pval, 4)})

ks_df = pd.DataFrame(ks_results).sort_values("ks_stat", ascending=False)
print("KS test (real vs generated) — lower stat = more similar:")
print(ks_df.to_string(index=False))

## 3. Volatility & Correlation

In [ ]:
# Correlation matrix comparison
corr_real = real_df.corr()
corr_gen  = gen_df.corr()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].imshow(corr_real, vmin=-1, vmax=1, cmap="RdBu_r")
axes[0].set_title("Real Correlation Matrix")
axes[0].set_xticks([]); axes[0].set_yticks([])
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(corr_gen, vmin=-1, vmax=1, cmap="RdBu_r")
axes[1].set_title("Generated Correlation Matrix")
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.colorbar(im1, ax=axes[1])

diff = corr_gen - corr_real
im2 = axes[2].imshow(diff, vmin=-0.5, vmax=0.5, cmap="RdBu_r")
axes[2].set_title("Difference (Gen - Real)")
axes[2].set_xticks([]); axes[2].set_yticks([])
plt.colorbar(im2, ax=axes[2])

plt.suptitle("Correlation Matrix: Real vs Generated", fontsize=14)
plt.tight_layout()
plt.savefig(REPO_ROOT / "artifacts" / "eval_correlation.png", dpi=150)
plt.show()

# Frobenius norm of difference
frob = np.linalg.norm(diff.values, "fro")
print(f"Frobenius norm of correlation diff: {frob:.4f}  (lower = better)")

In [ ]:
# Per-asset volatility comparison with bootstrap CI
vol_real = real_df.std()
vol_gen  = gen_df.std()

# Bootstrap CI on real vol
boot_vols = []
for _ in range(N_BOOTSTRAP):
    idx = np.random.choice(len(real_df), size=len(real_df), replace=True)
    boot_vols.append(real_df.iloc[idx].std())
boot_vols = pd.DataFrame(boot_vols)
vol_low  = boot_vols.quantile(0.025)
vol_high = boot_vols.quantile(0.975)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(asset_cols))
ax.bar(x - 0.2, vol_real, 0.35, label="Real", color="steelblue", alpha=0.8)
ax.bar(x + 0.2, vol_gen,  0.35, label="Generated", color="tomato", alpha=0.8)
ax.errorbar(x - 0.2, vol_real,
            yerr=[vol_real - vol_low, vol_high - vol_real],
            fmt="none", color="black", capsize=3, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(asset_cols, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Daily return std")
ax.set_title("Per-Asset Volatility: Real vs Generated (with 95% CI)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(REPO_ROOT / "artifacts" / "eval_volatility.png", dpi=150)
plt.show()